In [1]:
"""
变速拖船调度模型数据实例生成器 (Model 2: MTRSP-VS)
支持服务/转场速度分离的实例生成
拖船数量优化版 - 减少约35-40%拖船，避免过多闲置
"""

import numpy as np
import json
import os
from typing import Dict, List, Tuple


class VariableSpeedInstanceGenerator:
    """变速拖船调度实例生成器"""
    
    def __init__(self, seed: int = 42):
        """初始化生成器"""
        self.seed = seed
        self.instances = {}
        np.random.seed(seed)
        
        # 速度档位参数（调整为更大差距）
        self.speed_levels = {
            'slow': {'speed': 6.0, 'power_coef': 0.216},      # (6/10)^3 = 0.216
            'medium': {'speed': 10.0, 'power_coef': 1.000},   # (10/10)^3 = 1.000
            'fast': {'speed': 15.0, 'power_coef': 3.375}      # (15/10)^3 = 3.375
        }
        self.v_medium = 10.0  # 基准速度
    
    def generate_instance(self,
                         num_tasks: int,
                         num_tugs: int,
                         instance_name: str,
                         difficulty: str = 'medium',
                         area_size: float = 30.0,
                         T_max: float = 24.0) -> Dict:
        """生成单个实例"""
        
        print(f"生成: {instance_name:25s} 难度:{difficulty:8s} {num_tasks:3d}任务/{num_tugs:2d}拖船")
        
        # 获取难度参数
        diff_params = self._get_difficulty_params(difficulty, num_tasks)
        
        instance = {
            'metadata': {
                'name': instance_name,
                'num_tasks': num_tasks,
                'num_tugs': num_tugs,
                'area_size': area_size,
                'T_max': T_max,
                'M': 1000.0,
                'W': 100000.0,
                'difficulty': difficulty,
                'expected_completion_rate': diff_params['expected_rate'],
                'model_type': 'MTRSP-VS'
            },
            'base_location': [0.0, 0.0],
            'speed_levels': self.speed_levels
        }
        
        # 生成任务
        instance['tasks'] = self._generate_tasks(
            num_tasks, area_size, T_max, diff_params
        )
        
        # 生成拖船
        instance['tugboats'] = self._generate_tugboats(
            num_tugs, num_tasks, diff_params
        )
        
        # 只计算距离矩阵（其他矩阵由求解器动态计算）
        instance['distance_matrix'] = self._compute_distance_matrix(instance)
        
        self.instances[instance_name] = instance
        
        return instance
    
    def _get_difficulty_params(self, difficulty: str, num_tasks: int) -> Dict:
        """根据难度级别返回生成参数"""
        
        params = {
            'easy': {
                'multi_tug_prob': 0.15 if num_tasks <= 20 else 0.10,
                'time_window_width': 1.3,
                'fuel_multiplier': 1.2,
                'service_distance_range': (3.0, 8.0),
                'expected_rate': '85-95%'
            },
            'medium': {
                'multi_tug_prob': 0.25 if num_tasks <= 20 else 0.15,
                'time_window_width': 1.0,
                'fuel_multiplier': 1.0,
                'service_distance_range': (3.0, 9.0),
                'expected_rate': '75-85%'
            },
            'hard': {
                'multi_tug_prob': 0.35 if num_tasks <= 20 else 0.25,
                'time_window_width': 0.8,
                'fuel_multiplier': 0.9,
                'service_distance_range': (4.0, 10.0),
                'expected_rate': '65-75%'
            },
            'extreme': {
                'multi_tug_prob': 0.45 if num_tasks <= 20 else 0.35,
                'time_window_width': 0.6,
                'fuel_multiplier': 0.8,
                'service_distance_range': (5.0, 12.0),
                'expected_rate': '50-65%'
            }
        }
        
        return params.get(difficulty, params['medium'])
    
    def _generate_tasks(self, 
                       num_tasks: int,
                       area_size: float,
                       T_max: float,
                       diff_params: Dict) -> Dict:
        """生成任务数据"""
        
        tasks = {}
        task_types = ['靠泊', '离泊']
        
        total_time_span = T_max - 2.0
        
        for s in range(1, num_tasks + 1):
            # 任务类型
            task_type = np.random.choice(task_types)
            
            # 生成位置
            angle_offset = (s - 1) * (2 * np.pi / num_tasks)
            radius = np.random.uniform(area_size * 0.3, area_size * 0.45)
            
            start_loc = np.array([
                radius * np.cos(angle_offset) + np.random.uniform(-area_size*0.1, area_size*0.1),
                radius * np.sin(angle_offset) + np.random.uniform(-area_size*0.1, area_size*0.1)
            ])
            
            angle = np.random.uniform(0, 2*np.pi)
            
            # 服务距离
            service_distance = np.random.uniform(*diff_params['service_distance_range'])
            end_loc = start_loc + service_distance * np.array([np.cos(angle), np.sin(angle)])
            
            # 时间窗
            time_slot = total_time_span / num_tasks
            earliest_start = 0.5 + (s - 1) * time_slot
            
            time_window_width = diff_params['time_window_width']
            
            estimated_service_time = service_distance / 10.0
            
            max_latest_start = T_max - estimated_service_time * 1.2 - 0.5
            
            latest_start = min(earliest_start + time_window_width, max_latest_start)
            earliest_start = max(0.5, min(earliest_start, max_latest_start - 0.5))
            
            if latest_start <= earliest_start:
                latest_start = max_latest_start
                earliest_start = max(0.5, latest_start - time_window_width)
            
            # 需要的拖船数
            if np.random.random() < diff_params['multi_tug_prob']:
                num_tugs_needed = np.random.choice([2, 3], p=[0.8, 0.2])
            else:
                num_tugs_needed = 1
            
            # 最小马力需求
            if num_tugs_needed == 1:
                min_horsepower = np.random.randint(3000, 4500)
            elif num_tugs_needed == 2:
                min_horsepower = np.random.randint(6000, 9000)
            else:
                min_horsepower = np.random.randint(10000, 13000)
            
            tasks[str(s)] = {
                'type': task_type,
                'start_loc': start_loc.tolist(),
                'end_loc': end_loc.tolist(),
                'service_distance': round(service_distance, 2),
                'time_window': [round(earliest_start, 2), round(latest_start, 2)],
                'num_tugs_needed': int(num_tugs_needed),
                'min_horsepower': int(min_horsepower)
            }
        
        return tasks
    
    def _generate_tugboats(self, num_tugs: int, num_tasks: int, diff_params: Dict) -> Dict:
        """生成拖船数据"""
        
        tugboats = {}
        tug_names = ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon', 'Zeta', 
                     'Eta', 'Theta', 'Iota', 'Kappa', 'Lambda', 'Mu',
                     'Nu', 'Xi', 'Omicron', 'Pi', 'Rho', 'Sigma',
                     'Tau', 'Upsilon', 'Phi', 'Chi', 'Psi', 'Omega']
        
        for k in range(1, num_tugs + 1):
            # 马力
            horsepower = np.random.randint(4500, 6001)
            
            # 燃油容量
            base_fuel = 6000 if num_tasks <= 30 else (9000 if num_tasks <= 80 else 12000)
            fuel_capacity = int(base_fuel * diff_params['fuel_multiplier'])
            
            # 燃油系数
            alpha = round(np.random.uniform(0.15, 0.18), 3)
            beta = round(np.random.uniform(0.06, 0.08), 3)
            
            tugboats[str(k)] = {
                'name': tug_names[(k-1) % len(tug_names)],
                'horsepower': int(horsepower),
                'fuel_capacity': fuel_capacity,
                'alpha': alpha,
                'beta': beta
            }
        
        return tugboats
    
    def _compute_distance_matrix(self, instance: Dict) -> Dict:
        """计算距离矩阵"""
        
        tasks = instance['tasks']
        base = np.array(instance['base_location'])
        n = len(tasks)
        
        distance_matrix = {}
        
        for j in range(1, n + 1):
            start_loc = np.array(tasks[str(j)]['start_loc'])
            distance_matrix[f'0_{j}'] = float(np.linalg.norm(base - start_loc))
        
        for i in range(1, n + 1):
            end_i = np.array(tasks[str(i)]['end_loc'])
            for j in range(1, n + 1):
                if i != j:
                    start_j = np.array(tasks[str(j)]['start_loc'])
                    distance_matrix[f'{i}_{j}'] = float(np.linalg.norm(end_i - start_j))
        
        for i in range(1, n + 1):
            end_loc = np.array(tasks[str(i)]['end_loc'])
            distance_matrix[f'{i}_{n+1}'] = float(np.linalg.norm(end_loc - base))
        
        return distance_matrix
    
    def generate_all_instances(self):
        """生成128个不同难度的实例 - 拖船数量优化版"""
        
        print("\n" + "🚢 " * 20)
        print("变速拖船调度实例生成器 (Model 2: MTRSP-VS) - 128个实例")
        print("拖船数量优化版：减少35-40%拖船，提高资源利用率")
        print("🚢 " * 20 + "\n")
        
        instance_count = 0
        difficulties = ['easy', 'medium', 'hard', 'extreme']
        
        for difficulty in difficulties:
            print(f"\n{'='*80}")
            print(f"生成难度: {difficulty.upper()}")
            print(f"{'='*80}\n")
            
            # 小规模 (8个) - 拖船数量减少约40%
            # 原: 10/5, 15/6, 20/8, 30/12
            # 新: 10/3, 15/4, 20/5, 30/7
            for num_tasks, num_tugs, area in [(10, 3, 30), (15, 4, 35), (20, 5, 40), (30, 7, 45)]:
                for v in [1, 2]:
                    self.seed = 42 + instance_count
                    np.random.seed(self.seed)
                    prefix = 'X' if difficulty == 'extreme' else difficulty[0].upper()
                    self.generate_instance(
                        num_tasks=num_tasks,
                        num_tugs=num_tugs,
                        instance_name=f'VS_{prefix}_{num_tasks}_{num_tugs}_v{v}',
                        difficulty=difficulty,
                        area_size=area
                    )
                    instance_count += 1
            
            # 中规模 (12个) - 拖船数量减少约35%
            # 原: 40/15, 60/20, 80/25
            # 新: 40/10, 60/13, 80/16
            for num_tasks, num_tugs, area in [(40, 10, 50), (60, 13, 60), (80, 16, 70)]:
                for v in [1, 2, 3, 4]:
                    self.seed = 42 + instance_count
                    np.random.seed(self.seed)
                    prefix = 'X' if difficulty == 'extreme' else difficulty[0].upper()
                    self.generate_instance(
                        num_tasks=num_tasks,
                        num_tugs=num_tugs,
                        instance_name=f'VS_{prefix}_{num_tasks}_{num_tugs}_v{v}',
                        difficulty=difficulty,
                        area_size=area
                    )
                    instance_count += 1
            
            # 大规模 (12个) - 拖船数量减少约35%
            # 原: 100/30, 150/40, 200/50
            # 新: 100/20, 150/26, 200/33
            for num_tasks, num_tugs, area in [(100, 20, 80), (150, 26, 100), (200, 33, 120)]:
                for v in [1, 2, 3, 4]:
                    self.seed = 42 + instance_count
                    np.random.seed(self.seed)
                    prefix = 'X' if difficulty == 'extreme' else difficulty[0].upper()
                    self.generate_instance(
                        num_tasks=num_tasks,
                        num_tugs=num_tugs,
                        instance_name=f'VS_{prefix}_{num_tasks}_{num_tugs}_v{v}',
                        difficulty=difficulty,
                        area_size=area
                    )
                    instance_count += 1
        
        print(f"\n{'='*80}")
        print(f"总计生成 {instance_count} 个实例")
        print(f"{'='*80}")
    
    def save_instances(self, output_dir: str = 'instances_model2_vs'):
        """保存所有实例到JSON文件"""
        
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"\n{'='*80}")
        print(f"保存实例到目录: {output_dir}")
        print(f"{'='*80}\n")
        
        for name, instance in self.instances.items():
            filepath = os.path.join(output_dir, f"{name}.json")
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(instance, f, indent=2, ensure_ascii=False)
            print(f"✓ 保存: {filepath}")
        
        # 保存汇总信息
        summary = {
            'total_instances': len(self.instances),
            'model_type': 'MTRSP-VS (Variable Speed)',
            'speed_levels': self.speed_levels,
            'tugboat_reduction': '35-40% fewer tugboats than original',
            'difficulties': {
                'easy': len([n for n in self.instances if '_E_' in n]),
                'medium': len([n for n in self.instances if '_M_' in n]),
                'hard': len([n for n in self.instances if '_H_' in n]),
                'extreme': len([n for n in self.instances if '_X_' in n])
            },
            'instance_names': list(self.instances.keys())
        }
        
        summary_path = os.path.join(output_dir, 'summary.json')
        with open(summary_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        
        print(f"\n✓ 保存汇总信息: {summary_path}")


def main():
    """主函数"""
    
    generator = VariableSpeedInstanceGenerator(seed=42)
    generator.generate_all_instances()
    generator.save_instances(output_dir='instances_model2_vs')
    
    print("\n" + "="*80)
    print("实例生成完成！")
    print("="*80)
    print(f"\n所有 128 个实例已保存到 'instances_model2_vs' 目录")
    print(f"\n模型特点:")
    print(f"  • 变速优化: 3个速度档位 (slow=6节, medium=10节, fast=15节)")
    print(f"  • 服务/转场速度独立优化")
    print(f"  • 服务时长为决策变量（不再固定）")
    print(f"\n拖船数量优化:")
    print(f"  • 小规模: 减少约40% (例: 10任务 5拖船 → 3拖船)")
    print(f"  • 中规模: 减少约35% (例: 60任务 20拖船 → 13拖船)")
    print(f"  • 大规模: 减少约35% (例: 150任务 40拖船 → 26拖船)")
    print(f"\n难度分布:")
    print(f"  Easy (VS_E_):    32个 - 预期完成率 85-95%")
    print(f"  Medium (VS_M_):  32个 - 预期完成率 75-85%")
    print(f"  Hard (VS_H_):    32个 - 预期完成率 65-75%")
    print(f"  Extreme (VS_X_): 32个 - 预期完成率 50-65%")


if __name__ == "__main__":
    main()


🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 
变速拖船调度实例生成器 (Model 2: MTRSP-VS) - 128个实例
拖船数量优化版：减少35-40%拖船，提高资源利用率
🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 


生成难度: EASY

生成: VS_E_10_3_v1              难度:easy      10任务/ 3拖船
生成: VS_E_10_3_v2              难度:easy      10任务/ 3拖船
生成: VS_E_15_4_v1              难度:easy      15任务/ 4拖船
生成: VS_E_15_4_v2              难度:easy      15任务/ 4拖船
生成: VS_E_20_5_v1              难度:easy      20任务/ 5拖船
生成: VS_E_20_5_v2              难度:easy      20任务/ 5拖船
生成: VS_E_30_7_v1              难度:easy      30任务/ 7拖船
生成: VS_E_30_7_v2              难度:easy      30任务/ 7拖船
生成: VS_E_40_10_v1             难度:easy      40任务/10拖船
生成: VS_E_40_10_v2             难度:easy      40任务/10拖船
生成: VS_E_40_10_v3             难度:easy      40任务/10拖船
生成: VS_E_40_10_v4             难度:easy      40任务/10拖船
生成: VS_E_60_13_v1             难度:easy      60任务/13拖船
生成: VS_E_60_13_v2             难度:easy      60任务/13拖船
生成: VS_E_60_13_v3             难度:easy      60任务/13拖船
生成: VS_E_60_13_v4             难度:easy   